# DIP super-resolution with a PnP improvement

This notebook compares the assignment baseline (Deep Image Prior) with a plug-and-play (PnP) refinement inspired by Lecture 9. The PnP stage alternates bicubic data-consistency updates with TV denoising and gradually decreases the denoiser strength. It never uses the HR reference during optimization; the reference is used only for evaluation.

**Run the cells from top to bottom on a Colab GPU runtime.**

In [ ]:
# Colab setup. If you opened this notebook from GitHub, clone the repo first.
# In Colab: Runtime > Change runtime type > T4 GPU.
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "requirements.txt").exists():
    REPO_URL = "https://github.com/Dhadheechi/dip-sr.git"
    subprocess.check_call(["git", "clone", REPO_URL, "/content/dip-sr"])
    ROOT = Path("/content/dip-sr")

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")
])
sys.path.insert(0, str(ROOT / "src"))
print(f"repo: {ROOT}")

## W&B setup

W&B is required for the assignment run tracking. Create a free account at [wandb.ai](https://wandb.ai), create or choose a project name, and run the next cell. Colab will show a login link; paste the API key when prompted. Do not put the key in the notebook.

In [ ]:
import wandb

WANDB_PROJECT = "dip-sr-pnp"
USE_WANDB = True
if USE_WANDB:
    wandb.login()
print(f"W&B project: {WANDB_PROJECT}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

from dip_sr import (
    ExperimentConfig,
    PnPConfig,
    make_image_pair,
    psnr,
    run_experiment,
    run_pnp_refinement,
    tensor_to_image,
)

SEED = 42
SCALE = 4
image_path = ROOT / "data" / "sr" / "zebra_GT.png"
image = np.asarray(Image.open(image_path).convert("RGB"))
pair = make_image_pair(image, scale=SCALE)
print(f"device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
print(f"HR: {tuple(pair.hr.shape[-2:])}, LR: {tuple(pair.lr.shape[-2:])}")

## 1. Baseline DIP

This is the unchanged baseline objective: optimize a randomly initialized generator so that its bicubic downsample matches the LR observation. Keep this run and the PnP run at the same scale and seed.

In [ ]:
baseline_config = ExperimentConfig(
    scale=SCALE,
    iterations=2500,
    channels=64,
    seed=SEED,
    experiment_name="dip-baseline-zebra",
    wandb_project=WANDB_PROJECT if USE_WANDB else None,
    wandb_mode="online",
)
baseline = run_experiment(pair, baseline_config)
print(f"baseline HR PSNR: {psnr(baseline.output, pair.hr):.2f} dB")

## 2. PnP refinement

The continuation schedule starts with stronger TV denoising and reduces it over four stages. The data step uses the same differentiable bicubic operator as the baseline. These values are deliberately modest so the method preserves zebra edges while suppressing DIP artifacts.

In [ ]:
pnp_config = PnPConfig(
    outer_steps=4,
    inner_steps=8,
    data_step=0.8,
    denoise_relaxation=0.8,
    tv_weights=(0.08, 0.05, 0.03, 0.015),
    experiment_name="dip-pnp-tv-zebra",
    wandb_project=WANDB_PROJECT if USE_WANDB else None,
    wandb_mode="online",
)
pnp = run_pnp_refinement(pair, baseline.output, pnp_config)
print(f"PnP HR PSNR: {psnr(pnp.output, pair.hr):.2f} dB")
print(f"improvement: {psnr(pnp.output, pair.hr) - psnr(baseline.output, pair.hr):+.2f} dB")

In [ ]:
bicubic = torch.nn.functional.interpolate(
    pair.lr, size=pair.hr.shape[-2:], mode="bicubic", align_corners=False
)
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
images = [pair.hr, bicubic, baseline.output, pnp.output]
titles = [
    "HR reference",
    f"Bicubic ({psnr(bicubic, pair.hr):.2f} dB)",
    f"DIP baseline ({psnr(baseline.output, pair.hr):.2f} dB)",
    f"DIP + PnP ({psnr(pnp.output, pair.hr):.2f} dB)",
]
for axis, value, title in zip(axes, images, titles):
    axis.imshow(tensor_to_image(value))
    axis.set_title(title)
    axis.axis("off")
plt.tight_layout()
plt.show()

comparison = {
    "bicubic": psnr(bicubic, pair.hr),
    "dip_baseline": psnr(baseline.output, pair.hr),
    "dip_pnp_tv": psnr(pnp.output, pair.hr),
}
print(comparison)
if USE_WANDB:
    with wandb.init(project=WANDB_PROJECT, name="dip-sr-pnp-comparison", config=comparison) as comparison_run:
        comparison_run.log(comparison)

## Reproducing the result

1. Enable a Colab GPU and open this notebook from the GitHub repository.
2. Run the setup cell. It installs all Python dependencies, including W&B.
3. Create/sign in to W&B and paste the one-time API key when prompted.
4. Run the data cell, then run the baseline cell to completion.
5. Run the PnP cell and the comparison cell. The final printed `improvement` is the measured HR PSNR change.
6. Open the W&B project to compare `dip-baseline-zebra` and `dip-pnp-tv-zebra`.

The supplied HR image is used to create a reproducible synthetic bicubic LR observation. To test another image, replace `image_path` in the data cell; keep the same image, scale, seed, and iteration budget for both runs. For a real LR/HR pair, adapt `ImagePair` so that `pair.lr` is the observed LR image and `pair.hr` is evaluation-only ground truth.